# SO-101 Pi0.5 Pick/Place with PhysicalAI Runtime and OpenVINO

This notebook is a follow-up to the ACT getting-started notebook. It uses a more advanced Pi0.5 vision-language-action policy and shows how to deploy an exported PhysicalAI policy package with OpenVINO.

```text
Pi0.5 PhysicalAI policy package on Hugging Face
    -> InferenceModel.load(..., backend="openvino")
    -> PhysicalAI Runtime preprocessing from manifest.json
    -> OpenVINO CPU/GPU deployment benchmark
    -> LeRobot-style SO-101 replay visualization
```

The example uses:

- model package: [`eugene123tw/pi05-pick-place-purple-cube`](https://huggingface.co/eugene123tw/pi05-pick-place-purple-cube)
- replay dataset: [`gtamir/pick-place-purple-cube`](https://huggingface.co/datasets/gtamir/pick-place-purple-cube)
- robot/task: SO-101 pick-place-purple-cube with `top-cam` and `gripper-cam`


## Prerequisites

Install the required packages and clone the PhysicalAI Runtime and Physical AI Studio repositories.


In [ ]:
from pathlib import Path

requirements_file = Path("requirements.txt")
if not requirements_file.exists():
    requirements_file = Path("notebooks/requirements.txt")

if not Path("physicalai").exists():
    !git clone --depth 1 https://github.com/openvinotoolkit/physicalai.git
if not Path("physical-ai-studio").exists():
    !git clone --depth 1 https://github.com/open-edge-platform/physical-ai-studio.git

%pip install -q --extra-index-url https://download.pytorch.org/whl/cpu -r {requirements_file}
%pip install -q -e physicalai
%pip install -q -e "./physical-ai-studio/library[cpu]"


## 1. Configure the Notebook

Set repository IDs, local cache paths, OpenVINO Runtime, and SO-101 metadata. The notebook downloads only the files needed for replay: model package artifacts, dataset metadata, one episode parquet, and the two corresponding videos.

You can override defaults with environment variables:

- `PHYSICALAI_PI05_EXPORT_DIR`: local exported Pi0.5 policy package directory
- `PHYSICALAI_PI05_REPLAY_DATASET`: local LeRobot dataset root
- `PHYSICALAI_PI05_REPLAY_EPISODE`: replay episode id
- `PHYSICALAI_PI05_TASK`: language task prompt


In [ ]:
from pathlib import Path
import json
import os
import time

import cv2
import numpy as np
import openvino as ov
import openvino_tokenizers  # Registers custom tokenizer ops such as SpecialTokensSplit.
import pandas as pd
from huggingface_hub import hf_hub_download, snapshot_download
from physicalai.inference import InferenceModel

WORKSPACE = Path.cwd().resolve()
RUNTIME_ROOT = WORKSPACE / "physicalai"
ROOT = RUNTIME_ROOT
os.chdir(ROOT)

MODEL_REPO_ID = "eugene123tw/pi05-pick-place-purple-cube"
DATASET_REPO_ID = "gtamir/pick-place-purple-cube"
DATASET_NAME = "pick-place-purple-cube"
TASK_TEXT = os.environ.get("PHYSICALAI_PI05_TASK", "pick-place-purple-cube")
ASSETS_DIR = Path(os.environ.get("PHYSICALAI_ASSETS_DIR", WORKSPACE / "physicalai_assets")).resolve()

MODEL_DIR = Path(os.environ["PHYSICALAI_PI05_EXPORT_DIR"]).resolve() if os.environ.get("PHYSICALAI_PI05_EXPORT_DIR") else None
REPLAY_DATASET_DIR = Path(os.environ["PHYSICALAI_PI05_REPLAY_DATASET"]).resolve() if os.environ.get("PHYSICALAI_PI05_REPLAY_DATASET") else None
REPLAY_EPISODE_ID = int(os.environ.get("PHYSICALAI_PI05_REPLAY_EPISODE", "0"))
CACHE_DIR = ROOT / "exports" / "pi05_pick_place_purple_cube_cache"
VIS_DIR = ROOT / "exports" / "pi05_so101_visualization"

STATE_KEY = "state"
TASK_KEY = "task"
TOP_IMAGE_KEY = "images.top-cam"
GRIPPER_IMAGE_KEY = "images.gripper-cam"
TOP_DATASET_VIDEO_KEY = "observation.images.top-cam"
GRIPPER_DATASET_VIDEO_KEY = "observation.images.gripper-cam"

SO101_JOINT_ORDER = (
    "shoulder_pan",
    "shoulder_lift",
    "elbow_flex",
    "wrist_flex",
    "wrist_roll",
    "gripper",
)

core = ov.Core()
print("[INFO] OpenVINO:", ov.__version__)
print("[INFO] Available devices:", core.available_devices)
print("[INFO] Model repo:", MODEL_REPO_ID)
print("[INFO] Replay dataset repo:", DATASET_REPO_ID, "episode:", REPLAY_EPISODE_ID)
print("[INFO] Task:", TASK_TEXT)


## 2. Download the Pi0.5 PhysicalAI OpenVINO Package

This model repository already contains an exported PhysicalAI policy package:

- `manifest.json`: declares the runner, OpenVINO artifact, preprocessors, and postprocessors
- `pi05.xml` / `pi05.bin`: OpenVINO IR model
- `tokenizer.xml` / `tokenizer.bin`: OpenVINO tokenizer used by the Pi0.5 text prompt pipeline

The important deployment point is that `InferenceModel.load()` reads `manifest.json`, constructs the PhysicalAI Runtime preprocessing pipeline, compiles the OpenVINO model, and exposes `reset()`, `select_action()`, and `predict_action_chunk()`.


In [ ]:
if MODEL_DIR is None:
    MODEL_DIR = Path(snapshot_download(
        repo_id=MODEL_REPO_ID,
        local_dir=ASSETS_DIR / "models" / MODEL_REPO_ID.replace("/", "__"),
        allow_patterns=["manifest.json", "pi05.xml", "pi05.bin", "tokenizer.xml", "tokenizer.bin", "metadata.yaml", "README.md"],
        local_dir_use_symlinks=False,
    )).resolve()

required_model_files = ["manifest.json", "pi05.xml", "pi05.bin", "tokenizer.xml", "tokenizer.bin"]
missing = [name for name in required_model_files if not (MODEL_DIR / name).exists()]
if missing:
    raise FileNotFoundError(f"Missing required PhysicalAI package files in {MODEL_DIR}: {missing}")

manifest = json.loads((MODEL_DIR / "manifest.json").read_text(encoding="utf-8"))
print("[INFO] PhysicalAI package:", MODEL_DIR)
print("[INFO] Policy:", manifest.get("policy", {}).get("name"))
print("[INFO] Backend artifacts:", manifest.get("model", {}).get("artifacts", {}))
print("[INFO] Preprocessors:", [p.get("type") for p in manifest.get("model", {}).get("preprocessors", [])])
print("[INFO] Postprocessors:", [p.get("type") for p in manifest.get("model", {}).get("postprocessors", [])])


## 3. Optional: Export from a Pi0.5 Checkpoint

For day-to-day deployment, loading the already exported package above is the fastest path. If you have a Lightning checkpoint and Pi0.5 training dependencies installed, PhysicalAI Studio can export the checkpoint with the same API pattern used in the ACT notebook:

```python
from physicalai.policies.pi05 import Pi05

policy = Pi05.load_from_checkpoint("path/to/pi05.ckpt", compile_model=False)
policy.eval()
policy.export("./exports/pi05_policy", backend="openvino")
```

Pi0.5 is much larger than ACT, so this optional cell is disabled by default. Set `PHYSICALAI_RUN_PI05_EXPORT=1` if you want to try exporting the checkpoint from the model repo.


In [ ]:
RUN_CHECKPOINT_EXPORT = os.environ.get("PHYSICALAI_RUN_PI05_EXPORT", "0") == "1"

if RUN_CHECKPOINT_EXPORT:
    import subprocess
    import sys

    studio_library = str(WORKSPACE / "physical-ai-studio" / "library") + "[cpu,pi05]"
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", studio_library])

    from physicalai.policies.pi05 import Pi05

    checkpoint_path = Path(hf_hub_download(
        repo_id=MODEL_REPO_ID,
        filename="loss=0.1245.ckpt",
        local_dir=ASSETS_DIR / "checkpoints" / MODEL_REPO_ID.replace("/", "__"),
        local_dir_use_symlinks=False,
    )).resolve()
    export_from_checkpoint_dir = ROOT / "exports" / "pi05_pick_place_purple_cube_from_checkpoint"

    print("[STEP] Loading Pi0.5 checkpoint:", checkpoint_path)
    policy = Pi05.load_from_checkpoint(str(checkpoint_path), compile_model=False, map_location="cpu")
    policy.eval()

    print("[STEP] Exporting Pi0.5 checkpoint with PhysicalAI Studio API ...")
    policy.export(export_from_checkpoint_dir, backend="openvino")
    print("[DONE] Exported package:", export_from_checkpoint_dir)
else:
    print("[INFO] Skipping optional checkpoint export. Using the exported OpenVINO package from Hugging Face.")


## 4. Download One Replay Episode

The replay dataset is LeRobot-style. Instead of downloading the whole dataset up front, the notebook downloads metadata first, locates the selected episode, and then downloads only the corresponding parquet and camera videos.


In [ ]:
def resolve_dataset_root(path: Path) -> Path:
    path = Path(path).resolve()
    nested = path / DATASET_NAME
    return nested if nested.exists() else path

if REPLAY_DATASET_DIR is None:
    dataset_cache = ASSETS_DIR / "datasets" / DATASET_REPO_ID.replace("/", "__")
    snapshot_download(
        repo_id=DATASET_REPO_ID,
        repo_type="dataset",
        local_dir=dataset_cache,
        allow_patterns=[f"{DATASET_NAME}/meta/**"],
        local_dir_use_symlinks=False,
    )
    REPLAY_DATASET_DIR = resolve_dataset_root(dataset_cache)
else:
    REPLAY_DATASET_DIR = resolve_dataset_root(REPLAY_DATASET_DIR)

INFO_PATH = REPLAY_DATASET_DIR / "meta" / "info.json"
if not INFO_PATH.exists():
    raise FileNotFoundError(f"Missing LeRobot dataset metadata: {INFO_PATH}")

info = json.loads(INFO_PATH.read_text(encoding="utf-8"))
REPLAY_FPS = float(info.get("fps", 30))

print("[INFO] Replay dataset:", REPLAY_DATASET_DIR)
print("[INFO] Total episodes:", info.get("total_episodes"), "fps:", REPLAY_FPS)
print("[INFO] Video codec top-cam:", info["features"][TOP_DATASET_VIDEO_KEY]["info"].get("video.codec"))
print("[INFO] Video codec gripper-cam:", info["features"][GRIPPER_DATASET_VIDEO_KEY]["info"].get("video.codec"))


In [ ]:
def download_dataset_file(relative_path: str) -> Path:
    local_path = REPLAY_DATASET_DIR / relative_path
    if local_path.exists():
        return local_path
    downloaded = Path(hf_hub_download(
        repo_id=DATASET_REPO_ID,
        repo_type="dataset",
        filename=f"{DATASET_NAME}/{relative_path}",
        local_dir=REPLAY_DATASET_DIR.parent,
        local_dir_use_symlinks=False,
    ))
    return downloaded.resolve()

def load_episode_table(dataset_root: Path) -> pd.DataFrame:
    episode_files = sorted((dataset_root / "meta" / "episodes").glob("chunk-*/*.parquet"))
    if not episode_files:
        raise FileNotFoundError(f"No episode metadata files found under {dataset_root / 'meta' / 'episodes'}")
    return pd.concat([pd.read_parquet(path) for path in episode_files], ignore_index=True)

def load_replay_episode(dataset_root: Path, episode_id: int):
    episodes = load_episode_table(dataset_root)
    matches = episodes[episodes["episode_index"] == episode_id]
    if matches.empty:
        available = episodes["episode_index"].tolist()
        raise ValueError(f"Episode {episode_id} not found. Available episodes: {available[:10]} ... {available[-10:]}")
    episode_meta = matches.iloc[0]

    data_chunk = int(episode_meta["data/chunk_index"])
    data_file = int(episode_meta["data/file_index"])
    data_rel = f"data/chunk-{data_chunk:03d}/file-{data_file:03d}.parquet"
    parquet = download_dataset_file(data_rel)
    episode_df = pd.read_parquet(parquet)
    episode_df = episode_df[episode_df["episode_index"] == episode_id].reset_index(drop=True)

    def video_info(video_key: str):
        chunk_col = f"videos/{video_key}/chunk_index"
        file_col = f"videos/{video_key}/file_index"
        start_col = f"videos/{video_key}/from_timestamp"
        chunk_index = int(episode_meta[chunk_col]) if chunk_col in episode_meta.index else 0
        file_index = int(episode_meta[file_col]) if file_col in episode_meta.index else data_file
        start_seconds = float(episode_meta[start_col]) if start_col in episode_meta.index else 0.0
        video_rel = f"videos/{video_key}/chunk-{chunk_index:03d}/file-{file_index:03d}.mp4"
        video_path = download_dataset_file(video_rel)
        start_frame = int(round(start_seconds * REPLAY_FPS))
        return video_path, start_frame

    top_video, top_start = video_info(TOP_DATASET_VIDEO_KEY)
    gripper_video, gripper_start = video_info(GRIPPER_DATASET_VIDEO_KEY)
    return episode_df, top_video, gripper_video, top_start, gripper_start

episode_df, top_video_path, gripper_video_path, TOP_VIDEO_START_FRAME, GRIPPER_VIDEO_START_FRAME = load_replay_episode(
    REPLAY_DATASET_DIR,
    REPLAY_EPISODE_ID,
)

print("[INFO] Episode:", REPLAY_EPISODE_ID, "rows:", len(episode_df))
print("[INFO] Top video:", top_video_path, "start_frame:", TOP_VIDEO_START_FRAME)
print("[INFO] Gripper video:", gripper_video_path, "start_frame:", GRIPPER_VIDEO_START_FRAME)
print(episode_df.head(2))


## 5. PhysicalAI Runtime: Load, Reset, Select Action

`InferenceModel.load()` is the key PhysicalAI Runtime API for deployment. For this Pi0.5 package it automatically wires together:

- quantile normalization for SO-101 state
- Pi0.5 image resize and prompt construction
- OpenVINO tokenizer inference
- OpenVINO Pi0.5 policy inference
- action denormalization back to SO-101 joint space

The input observation stays application-friendly: state, camera frames, and a task string.


In [ ]:
def image_to_bhwc_float(frame_rgb: np.ndarray) -> np.ndarray:
    return frame_rgb.astype(np.float32)[None, ...] / 255.0

def as_action_vector(values, name="action") -> np.ndarray:
    arr = np.asarray(values, dtype=np.float32)
    if arr.ndim == 0:
        raise ValueError(f"Expected {name} to be a vector, got scalar value {arr!r}")
    if arr.ndim > 1:
        arr = arr.reshape(-1, arr.shape[-1])[0]
    arr = arr[: len(SO101_JOINT_ORDER)].astype(np.float32)
    if arr.shape[0] != len(SO101_JOINT_ORDER):
        raise ValueError(f"Expected {name} length {len(SO101_JOINT_ORDER)}, got shape {arr.shape}")
    return arr

def make_policy_observation(top_frame: np.ndarray, gripper_frame: np.ndarray, state: np.ndarray) -> dict:
    return {
        STATE_KEY: as_action_vector(state, name="observation.state")[None, :],
        TOP_IMAGE_KEY: image_to_bhwc_float(top_frame),
        GRIPPER_IMAGE_KEY: image_to_bhwc_float(gripper_frame),
        TASK_KEY: [TASK_TEXT],
    }

def openvino_config_for_device(device: str) -> dict[str, str]:
    return {"CACHE_DIR": str(CACHE_DIR)}


## 6. Select Deployment Device

Choose the OpenVINO target device. Pi0.5 is substantially larger than ACT, so CPU is the safest default. If a compatible Intel GPU is available, you can select it for acceleration.


In [ ]:
import ipywidgets as widgets
from IPython.display import display

device_options = list(core.available_devices)
default_device = "GPU" if "GPU" in device_options else "CPU"
TARGET_DEVICE = widgets.Dropdown(
    options=device_options,
    value=default_device if default_device in device_options else device_options[0],
    description="Device:",
)
display(TARGET_DEVICE)


In [ ]:
def read_video_rgb(path: Path, max_frames: int | None = None, start_frame: int = 0):
    cap = cv2.VideoCapture(str(path))
    if start_frame:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(start_frame))
    frames = []
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)
        if max_frames is not None and len(frames) >= max_frames:
            break
    cap.release()
    if frames:
        return frames

    print(f"[WARN] OpenCV could not decode {path.name}; retrying with imageio-ffmpeg.")
    try:
        import imageio
        reader = imageio.get_reader(str(path), "ffmpeg")
        try:
            for frame_index, frame in enumerate(reader):
                if frame_index < start_frame:
                    continue
                frames.append(np.asarray(frame[..., :3], dtype=np.uint8))
                if max_frames is not None and len(frames) >= max_frames:
                    break
        finally:
            reader.close()
    except Exception as exc:
        raise RuntimeError(
            "Could not decode replay video. This dataset uses AV1 video; install imageio-ffmpeg "
            "or re-encode the validation videos to H.264 for wider platform support."
        ) from exc

    if not frames:
        raise RuntimeError(f"Decoded zero frames from {path}; check video codec support and start_frame={start_frame}.")
    return frames

def benchmark_physicalai_pi05(device: str, runs: int = 5):
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    max_frames = 1
    top_frames = read_video_rgb(top_video_path, max_frames=max_frames, start_frame=TOP_VIDEO_START_FRAME)
    gripper_frames = read_video_rgb(gripper_video_path, max_frames=max_frames, start_frame=GRIPPER_VIDEO_START_FRAME)
    obs = make_policy_observation(top_frames[0], gripper_frames[0], episode_df.iloc[0]["observation.state"])

    config = openvino_config_for_device(device)
    start = time.perf_counter()
    model = InferenceModel.load(MODEL_DIR, backend="openvino", device=device, **config)
    load_ms = (time.perf_counter() - start) * 1000

    model.reset()
    _ = model.predict_action_chunk(obs)
    first_action = model.select_action(obs)
    timings = []
    for _ in range(runs):
        model.reset()
        start = time.perf_counter()
        chunk = model.predict_action_chunk(obs)
        timings.append((time.perf_counter() - start) * 1000)

    return model, {
        "device": device,
        "load_ms": load_ms,
        "avg_ms": float(np.mean(timings)),
        "p50_ms": float(np.percentile(timings, 50)),
        "p95_ms": float(np.percentile(timings, 95)),
        "fps": float(1000 / np.mean(timings)),
        "chunk_shape": tuple(chunk.shape),
        "select_action_shape": tuple(np.asarray(first_action).shape),
    }

def benchmark_with_fallback(device: str, runs: int = 5):
    try:
        return benchmark_physicalai_pi05(device, runs=runs)
    except RuntimeError as exc:
        print(f"[WARN] PhysicalAI Pi0.5 OpenVINO failed on {device}: {type(exc).__name__}: {exc}")
        if device != "CPU":
            print("[INFO] Falling back to CPU so the validation can continue.")
            return benchmark_physicalai_pi05("CPU", runs=runs)
        raise

physicalai_model, selected_result = benchmark_with_fallback(TARGET_DEVICE.value, runs=5)
print("[RESULT] PhysicalAI Pi0.5 OpenVINO deployment")
for key, value in selected_result.items():
    print(f"  {key}: {value}")


## 7. Replay Visualization

Replay a recorded SO-101 episode and overlay the Pi0.5/OpenVINO actions.

- Left: recorded `top-cam` and `gripper-cam` views.
- Right: a joint-space sketch of the observed state and predicted target.
- Bottom: predicted action vs dataset expert action for each joint.

Replay MAE is an offline domain-match sanity check. It is useful for confirming that the exported policy behaves consistently on a recorded validation episode, but real task quality still requires closed-loop robot or simulator evaluation.


In [ ]:
from PIL import Image, ImageDraw
from IPython.display import Image as IPyImage, display as ipy_display

def so101_points(values, origin=(590, 330), scale=1.0):
    vals = np.asarray(values, dtype=np.float32)
    lengths = np.array([70, 58, 48, 34], dtype=np.float32) * scale
    angles = np.deg2rad([
        -90 + vals[0] * 0.55,
        vals[1] * 0.45,
        vals[2] * 0.35,
        vals[3] * 0.25 + vals[4] * 0.08,
    ])
    pts = [np.array(origin, dtype=np.float32)]
    heading = 0.0
    for length, angle in zip(lengths, angles):
        heading += angle
        pts.append(pts[-1] + np.array([np.cos(heading), np.sin(heading)]) * length)
    return [(int(x), int(y)) for x, y in pts]

def draw_arm(draw, values, color, width=9):
    pts = so101_points(values)
    for a, b in zip(pts[:-1], pts[1:]):
        draw.line([a, b], fill=color, width=width)
        draw.line([a, b], fill=(245, 248, 250), width=max(2, width // 3))
    for p in pts:
        draw.ellipse([p[0] - 7, p[1] - 7, p[0] + 7, p[1] + 7], fill=(20, 24, 30), outline=color, width=3)
    gripper = float(np.asarray(values)[5])
    ee = pts[-1]
    span = int(8 + np.clip(gripper, 0, 100) * 0.12)
    draw.line([(ee[0] - span, ee[1] - 9), (ee[0] + span, ee[1] + 9)], fill=color, width=4)

def draw_bars(draw, x, y, width, height, pred, expert):
    row_h = height // len(SO101_JOINT_ORDER)
    center = x + width // 2
    draw.line([(center, y), (center, y + height)], fill=(80, 90, 102), width=1)
    for i, name in enumerate(SO101_JOINT_ORDER):
        yy = y + i * row_h + 6
        draw.text((x, yy), name, fill=(220, 226, 234))
        for val, color, offset in [(expert[i], (95, 170, 255), 13), (pred[i], (255, 190, 85), 28)]:
            v = float(np.clip(val, -100, 100))
            bar = int((v / 100.0) * (width * 0.32))
            if bar >= 0:
                draw.rectangle([center, yy + offset, center + bar, yy + offset + 8], fill=color)
            else:
                draw.rectangle([center + bar, yy + offset, center, yy + offset + 8], fill=color)
        draw.text((x + width - 105, yy + 10), f"E {expert[i]:6.1f}", fill=(95, 170, 255))
        draw.text((x + width - 105, yy + 27), f"P {pred[i]:6.1f}", fill=(255, 190, 85))

def make_overlay_frame(top, gripper, state, pred, expert, frame_idx, latency_ms, device):
    canvas = Image.new("RGB", (1120, 720), (18, 22, 28))
    top_img = Image.fromarray(top).resize((512, 288))
    grip_img = Image.fromarray(gripper).resize((512, 288))
    draw_top = ImageDraw.Draw(top_img)
    draw_grip = ImageDraw.Draw(grip_img)
    draw_top.rectangle([0, 0, 92, 26], fill=(0, 0, 0))
    draw_top.text((8, 6), "top-cam", fill=(255, 255, 255))
    draw_grip.rectangle([0, 0, 122, 26], fill=(0, 0, 0))
    draw_grip.text((8, 6), "gripper-cam", fill=(255, 255, 255))
    canvas.paste(top_img, (20, 72))
    canvas.paste(grip_img, (20, 374))

    draw = ImageDraw.Draw(canvas)
    draw.text((20, 24), "SO-101 Pick/Place Replay: PhysicalAI + OpenVINO Pi0.5", fill=(242, 246, 250))
    draw.text((20, 48), f"task={TASK_TEXT} | device={device} | frame={frame_idx:03d} | latency={latency_ms:.1f} ms", fill=(170, 184, 199))

    draw.rectangle([560, 72, 1098, 430], outline=(65, 76, 90), width=2)
    draw.text((580, 92), "Joint-space viewer", fill=(235, 241, 245))
    draw.text((580, 116), "blue: observed state   amber: Pi0.5 predicted target", fill=(170, 184, 199))
    draw_arm(draw, state, (95, 170, 255), width=11)
    draw_arm(draw, pred, (255, 190, 85), width=7)

    err = float(np.mean(np.abs(pred - expert)))
    draw.text((580, 398), f"mean |pred - expert| = {err:.2f}", fill=(235, 241, 245))

    draw.rectangle([560, 454, 1098, 704], outline=(65, 76, 90), width=2)
    draw.text((580, 468), "Action comparison in SO-101 normalized joint space", fill=(235, 241, 245))
    draw_bars(draw, 580, 500, 490, 186, pred, expert)
    return canvas

def describe_replay_mae(mean_mae: float) -> str:
    if mean_mae < 5.0:
        return "low offline MAE on this replay dataset; export/runtime and replay domain look consistent"
    if mean_mae < 15.0:
        return "moderate offline MAE; inspect the overlay and confirm this replay matches the training domain"
    return "high offline MAE; often expected when the checkpoint was trained on a different dataset, camera setup, task, or action distribution"

def run_replay_visualization(max_rendered_frames=120, render_stride=3):
    max_replay_steps = max_rendered_frames * render_stride
    top_frames = read_video_rgb(top_video_path, max_frames=max_replay_steps, start_frame=TOP_VIDEO_START_FRAME)
    gripper_frames = read_video_rgb(gripper_video_path, max_frames=max_replay_steps, start_frame=GRIPPER_VIDEO_START_FRAME)
    n = min(len(top_frames), len(gripper_frames), len(episode_df), max_replay_steps)
    print("[INFO] Loaded frames:", len(top_frames), len(gripper_frames), "using", n)
    if n == 0:
        raise RuntimeError(
            "No replay frames were decoded. The validation videos may require AV1 decode support; "
            "try installing imageio-ffmpeg or re-encoding the videos to H.264."
        )

    replay_device = selected_result["device"] if "selected_result" in globals() else TARGET_DEVICE.value
    model = physicalai_model if "physicalai_model" in globals() and selected_result["device"] == replay_device else InferenceModel.load(
        MODEL_DIR,
        backend="openvino",
        device=replay_device,
        **openvino_config_for_device(replay_device),
    )
    model.reset()

    vis_frames = []
    latencies = []
    errors = []
    predictions = []
    expert_actions = []

    for frame_idx in range(n):
        state = as_action_vector(episode_df.iloc[frame_idx]["observation.state"], name="observation.state")
        expert = as_action_vector(episode_df.iloc[frame_idx]["action"], name="expert action")
        top = top_frames[frame_idx]
        gripper = gripper_frames[frame_idx]
        obs = make_policy_observation(top, gripper, state)

        start = time.perf_counter()
        raw_pred = model.select_action(obs)
        pred = as_action_vector(raw_pred, name="predicted action")
        latency_ms = (time.perf_counter() - start) * 1000
        if frame_idx == 0:
            print("[INFO] Replay action shapes:", {
                "raw_pred": tuple(np.asarray(raw_pred).shape),
                "pred": tuple(pred.shape),
                "expert": tuple(expert.shape),
            })
        latencies.append(latency_ms)
        errors.append(float(np.mean(np.abs(pred - expert))))
        predictions.append(pred)
        expert_actions.append(expert)
        if frame_idx % render_stride == 0 and len(vis_frames) < max_rendered_frames:
            vis_frames.append(make_overlay_frame(top, gripper, state, pred, expert, frame_idx, latency_ms, replay_device))

    VIS_DIR.mkdir(parents=True, exist_ok=True)
    gif_path = VIS_DIR / "so101_pick_place_pi05_openvino.gif"
    vis_frames[0].save(gif_path, save_all=True, append_images=vis_frames[1:], duration=66, loop=0)
    mean_mae = float(np.mean(errors))
    per_joint_mae = np.mean(np.abs(np.stack(predictions) - np.stack(expert_actions)), axis=0)
    print("[RESULT] Replay steps:", len(predictions))
    print("[RESULT] Rendered frames:", len(vis_frames))
    print("[RESULT] Avg select_action latency ms:", float(np.mean(latencies)))
    print("[RESULT] Avg MAE vs expert action:", mean_mae)
    print("[RESULT] Per-joint MAE:", dict(zip(SO101_JOINT_ORDER, per_joint_mae.round(3).tolist())))
    print("[INTERPRETATION]", describe_replay_mae(mean_mae))
    print("[INTERPRETATION] Replay MAE is a domain-match sanity check, not an OpenVINO correctness test.")
    print("[DONE] Saved GIF:", gif_path)
    ipy_display(IPyImage(filename=str(gif_path)))

run_replay_visualization(max_rendered_frames=120, render_stride=3)


## 8. Optional: Run Pi0.5 on Physical SO-101 Hardware

After the replay sanity check, you can connect the same OpenVINO Pi0.5 policy package to real cameras and an SO-101 follower arm. This section follows the runtime pattern from `end_to_end_demo.ipynb`:

1. configure the robot serial port, calibration file, and camera names,
2. optionally install hardware extras,
3. discover camera devices,
4. load the PhysicalAI OpenVINO policy,
5. connect cameras and robot,
6. run `PolicyRuntime` for a short controlled session.

Safety notes before running on hardware:

- Keep the robot workspace clear and keep power/USB access reachable.
- Use the same camera names as training: `top-cam` and `gripper-cam`.
- Use a calibrated SO-101 follower arm; uncalibrated raw ticks are not suitable for policy deployment.
- The live run cells are guarded by `RUN_LIVE_ROBOT = False` by default. Change it to `True` only when the hardware is ready.


### 8.1 Configure Live Hardware

Adjust these values for your machine. On Windows, SO-101 serial ports usually look like `COM3`; on Linux they usually look like `/dev/ttyACM0` or `/dev/ttyUSB0`.


In [ ]:
def parse_camera_device(value: str):
    value = str(value)
    return int(value) if value.isdecimal() else value

RUN_LIVE_ROBOT = False
INSTALL_HARDWARE_DEPS = RUN_LIVE_ROBOT

# OpenVINO target for live policy inference.
LIVE_DEVICE = selected_result["device"] if "selected_result" in globals() else TARGET_DEVICE.value

# Robot configuration.
SO101_PORT = os.environ.get("PHYSICALAI_SO101_PORT", "COM3" if os.name == "nt" else "/dev/ttyACM0")
SO101_CALIBRATION = os.environ.get(
    "PHYSICALAI_SO101_CALIBRATION",
    str(Path.home() / ".cache" / "physicalai" / "robots" / "<robot-id>" / "calibrations" / "<cal-id>.json"),
)

# Camera configuration. Names must match the Pi0.5 package and training dataset.
CAMERAS = [
    ("top-cam", "uvc", parse_camera_device(os.environ.get("PHYSICALAI_TOP_CAMERA", "0"))),
    ("gripper-cam", "uvc", parse_camera_device(os.environ.get("PHYSICALAI_GRIPPER_CAMERA", "1"))),
]
CAMERA_WIDTH = int(os.environ.get("PHYSICALAI_CAMERA_WIDTH", "640"))
CAMERA_HEIGHT = int(os.environ.get("PHYSICALAI_CAMERA_HEIGHT", "480"))
CAMERA_FPS = int(os.environ.get("PHYSICALAI_CAMERA_FPS", "30"))

# Runtime configuration. Start short while validating a new setup.
LIVE_FPS = int(os.environ.get("PHYSICALAI_LIVE_FPS", "30"))
LIVE_DURATION_S = float(os.environ.get("PHYSICALAI_LIVE_DURATION_S", "10"))

print("[INFO] RUN_LIVE_ROBOT:", RUN_LIVE_ROBOT)
print("[INFO] INSTALL_HARDWARE_DEPS:", INSTALL_HARDWARE_DEPS)
print("[INFO] LIVE_DEVICE:", LIVE_DEVICE)
print("[INFO] SO101_PORT:", SO101_PORT)
print("[INFO] SO101_CALIBRATION:", SO101_CALIBRATION)
print("[INFO] CAMERAS:", CAMERAS)
print("[INFO] TASK_TEXT:", TASK_TEXT)


### 8.2 Optional Hardware Dependencies

The replay path does not need robot hardware packages. A live SO-101 session needs the `transport` extra for `SharedCamera` and the `so101` extra for the Feetech servo SDK. If you use RealSense or Basler cameras, install the corresponding `physicalai[realsense]` or `physicalai[basler]` extra as well.


In [ ]:
if INSTALL_HARDWARE_DEPS:
    import subprocess
    import sys

    runtime_with_hardware_extras = str(WORKSPACE / "physicalai") + "[transport,so101]"
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", runtime_with_hardware_extras])
    print("[DONE] Installed PhysicalAI hardware extras: transport, so101")
else:
    print("[SKIP] Hardware extras are not installed. Set INSTALL_HARDWARE_DEPS = True if this environment needs them.")


### 8.3 Discover Cameras

Run this cell to list visible camera devices. Use the printed device IDs in the hardware configuration cell above.


In [ ]:
from physicalai.capture import discover_all

for driver, devices in discover_all().items():
    if not devices:
        continue
    print(f"\n[{driver}]")
    for dev in devices:
        print(f"  {dev.device_id}  -  {dev.name}")


### 8.4 Load the Live Policy

This loads the same PhysicalAI OpenVINO package used for replay. `openvino_tokenizers` must be imported before reading `tokenizer.xml` because the tokenizer IR uses OpenVINO Tokenizers custom ops.


In [ ]:
import openvino_tokenizers  # Registers OpenVINO tokenizer custom ops.
from physicalai.inference import InferenceModel

LIVE_CACHE_DIR = ROOT / "exports" / "pi05_live_cache"
LIVE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

live_policy = InferenceModel.load(
    MODEL_DIR,
    backend="openvino",
    device=LIVE_DEVICE,
    CACHE_DIR=str(LIVE_CACHE_DIR),
)
live_policy.reset()
print(f"[INFO] Live policy loaded on {LIVE_DEVICE}: {live_policy}")


### 8.5 Connect Cameras and SO-101

This cell creates `SharedCamera` subscribers for the two live camera streams and connects the calibrated SO-101 follower arm. It is skipped unless `RUN_LIVE_ROBOT` is `True`.


In [ ]:
from physicalai.capture import SharedCamera
from physicalai.robot import SO101

live_cameras = {}
live_robot = None

if not RUN_LIVE_ROBOT:
    print("[SKIP] Set RUN_LIVE_ROBOT = True in the hardware config cell when the robot and cameras are ready.")
else:
    try:
        for name, driver, device_id in CAMERAS:
            kwargs = {"serial_number": device_id} if driver == "realsense" else {"device": device_id}
            cam = SharedCamera(
                driver,
                **kwargs,
                width=CAMERA_WIDTH,
                height=CAMERA_HEIGHT,
                fps=CAMERA_FPS,
            )
            cam.connect()
            live_cameras[name] = cam
            print(f"[OK] Camera '{name}' connected from {driver}:{device_id}")

        live_robot = SO101(port=SO101_PORT, calibration=SO101_CALIBRATION, role="follower")
        live_robot.connect()
        print(f"[OK] SO-101 follower connected on {SO101_PORT}")
    except Exception:
        for cam in live_cameras.values():
            try:
                cam.disconnect()
            except Exception:
                pass
        live_cameras.clear()
        if live_robot is not None:
            try:
                live_robot.disconnect()
            except Exception:
                pass
        live_robot = None
        raise


### 8.6 Run the Policy Runtime

Pi0.5 requires a language task prompt. The current `PolicyRuntime` builds robot state and camera image inputs, so this notebook adds a tiny `TaskPolicyRuntime` subclass that injects `task=[TASK_TEXT]` before inference. All scheduling, action queueing, and robot command sending still use the PhysicalAI Runtime APIs.


In [ ]:
from typing import Any

from physicalai.runtime import ActionQueue, AsyncExecution, PolicyRuntime

class TaskPolicyRuntime(PolicyRuntime):
    def __init__(self, *args: Any, task: str, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        self._task = task

    def _build_model_input(self) -> dict[str, Any]:
        model_input = super()._build_model_input()
        model_input["task"] = [self._task]
        return model_input

if not RUN_LIVE_ROBOT:
    print("[SKIP] Live runtime is disabled. Set RUN_LIVE_ROBOT = True to run on hardware.")
elif live_robot is None or not live_cameras:
    raise RuntimeError("Robot/cameras are not connected. Run the connection cell first.")
else:
    live_policy.reset()
    runtime = TaskPolicyRuntime(
        robot=live_robot,
        model=live_policy,
        execution=AsyncExecution(threshold=0.5, fps=LIVE_FPS),
        fps=LIVE_FPS,
        cameras=live_cameras,
        action_queue=ActionQueue(),
        task=TASK_TEXT,
    )

    print(f"[RUN] Running Pi0.5 policy at {LIVE_FPS} fps for {LIVE_DURATION_S:.1f}s; task={TASK_TEXT!r}")
    with runtime:
        stats = runtime.run(duration_s=LIVE_DURATION_S)

    print(
        f"[DONE] {stats.steps} steps | "
        f"{stats.inference_count} inferences | "
        f"{stats.total_pops} action pops | "
        f"{stats.total_holds} holds"
    )


### 8.7 Disconnect Hardware

Always run this cell after a live session or if a connection attempt fails midway.


In [ ]:
if "live_cameras" in globals():
    for name, cam in list(live_cameras.items()):
        try:
            cam.disconnect()
            print(f"[OK] Camera '{name}' disconnected")
        except Exception as exc:
            print(f"[WARN] Camera '{name}' disconnect failed: {type(exc).__name__}: {exc}")
    live_cameras.clear()

if "live_robot" in globals() and live_robot is not None:
    try:
        live_robot.disconnect()
        print("[OK] SO-101 disconnected")
    except Exception as exc:
        print(f"[WARN] Robot disconnect failed: {type(exc).__name__}: {exc}")
    live_robot = None
